In [1]:
from astropy.io import fits
import numpy as np
from astropy.table import Table
import pandas as pd
import glob
from astropy.table import vstack
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.backends.backend_pdf import PdfPages
import os

In [2]:
df = pd.read_csv("../Class_wise_v4/Halpha_emitter_wise_noise.csv")
len(df)

1750

In [3]:
# Colors
df['W1_W2'] = df['W1mag'] - df['W2mag']
df['J_H'] = df['Jmag'] - df['Hmag']
df['H_W2'] = df['Hmag'] - df['W2mag']
df['Ks_W3'] = df['Kmag'] - df['W3mag']
df['W1_W4'] = df['W1mag'] - df['W4mag']
df['r_i'] = df['rImag'] - df['imag']
df['r_Ha'] = df['rImag'] - df['Hamag']



Akras et al. (2019):

Three different classification tree models were found that identify
PNe in the best possible way: W1 − W4 ≥ 7.87 and J − H < 1.10
(M1); H − W2 ≥ 2.24 and J − H < 0.50 (M2); and Ks − W3 ≥
6.42 and J − H < 1.31 (M3).

In [4]:
# Modelo M1 (W1-W4 y J-H)
mask_m1 = ((df['W1_W4'] >= 7.87) & (df['J_H'] < 1.10))
    
# Modelo M2 (H-W2 y J-H) - ¡Tu preferido!
mask_m2 = ((df['H_W2'] >= 2.24) & (df['J_H'] < 0.50))
    
# Modelo M3 (Ks-W3 y J-H)
mask_m3 = ((df['Ks_W3'] >= 6.42) & (df['J_H'] < 1.31))

In [5]:
# DataFrames separados
df_m1 = df[mask_m1].copy()
df_m1['model'] = 'M1'
    
df_m2 = df[mask_m2].copy()
df_m2['model'] = 'M2'
    
df_m3 = df[mask_m3].copy()
df_m3['model'] = 'M3'

In [6]:
df_full_PN = pd.concat([df_m1, df_m2, df_m3])
df_full_PN

,Name,RAJ2000,DEJ2000,GLON,GLAT,SourceID,ePos,Class,pStar,pGalaxy,...,PC5,Label,W1_W2,J_H,H_W2,Ks_W3,W1_W4,r_i,r_Ha,model
317,J211937.22+545328.8,319.905078,54.891340,95.544481,3.707010,477336-1-8163,0.034,99.0,0.05,0.95,...,6.159535,-1,2.058,0.707,5.851,9.167,8.593,0.76,1.22,M1
326,J213646.48+562716.6,324.193667,56.454608,98.377140,3.165845,478602-1-4135,0.036,-1.0,1.00,0.00,...,5.922983,-1,2.283,0.319,4.175,8.573,8.838,0.60,1.07,M1
521,J064724.25-000137.8,101.851057,-0.027175,212.496251,-0.909865,650202-4-3360,0.032,-1.0,1.00,0.00,...,5.908355,-1,1.097,0.487,2.281,6.596,8.286,0.49,0.57,M1
566,J063122.56+045019.5,97.844016,4.838763,206.335101,-2.238908,374770-1-743,0.035,99.0,1.00,0.00,...,5.494996,-1,0.299,0.840,1.071,5.188,9.165,1.05,0.77,M1
577,J063149.26+045700.9,97.955238,4.950254,206.287253,-2.089161,429747-2-2779,0.053,-1.0,1.00,0.00,...,5.951644,-1,0.543,0.853,1.906,4.889,8.324,0.86,0.65,M1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1576,J221952.97+561449.7,334.970710,56.247137,102.979530,-0.637602,462337-3-6954,0.034,99.0,0.95,0.05,...,5.658783,-1,0.600,0.245,2.324,6.819,7.446,0.53,0.49,M3
1599,J232247.53+595217.2,350.698023,59.871442,112.015505,-1.108140,476438-4-4802,0.030,-1.0,1.00,0.00,...,5.960554,-1,0.480,0.716,2.586,6.636,8.406,0.56,0.46,M3
1645,J231541.23+620654.4,348.921805,62.115113,111.990056,1.294818,635580-2-2824,0.030,-1.0,1.00,0.00,...,6.214386,-1,0.870,1.184,3.824,6.599,6.397,0.39,0.49,M3
1712,J194557.22+243745.6,296.488437,24.629325,60.873379,-0.034522,364585-1-4405,0.028,-1.0,1.00,0.00,...,6.215732,-1,1.459,0.943,3.984,6.967,7.600,0.64,0.77,M3


In [7]:
# Ejemplo: Conservar la última aparición
df_full_PN_unique = df_full_PN.drop_duplicates(subset=['Name'], keep='last')

In [10]:
df_full_PN_unique[["RAJ2000", "DEJ2000", "rImag"]]

,RAJ2000,DEJ2000,rImag
566,97.844016,4.838763,17.54
577,97.955238,4.950254,16.29
581,97.916177,4.944209,18.13
583,97.923491,5.026523,19.06
721,99.121919,10.833720,16.51
989,288.178775,5.762863,17.65
1037,82.035785,34.427328,16.22
1225,3.851996,61.238380,16.33
1344,42.693960,60.437394,17.58
1428,34.285149,63.269821,17.84


In [9]:
# 9. Guardar para crossmatch con LAMOST
df_full_PN.to_csv('../Class_wise_v4/PNe_candidates_noise_Akras.csv', index=False)